In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
import open_clip
from splice.model import SPLICE
import torchvision.transforms.functional as TF
from collections import defaultdict
import os
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import sys
sys.path.append("/workspaces/Object-Centric-Concept-Bottlenecks")
from utils import draw_circle_on_image

print("Imports OK")

In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
import open_clip
from splice.model import SPLICE
import torchvision.transforms.functional as TF
from collections import defaultdict
import os
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import sys
sys.path.append("/workspaces/Object-Centric-Concept-Bottlenecks")
from utils import draw_circle_on_image

print("Imports OK")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

concepts = torch.load("/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt", map_location=device)
image_mean = torch.load("/root/.cache/splice/means/open_clip_ViT-B-32_image.pt", map_location=device)
clip_backbone, _, splice_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_backbone = clip_backbone.to(device)

splice_model = SPLICE(image_mean=image_mean, dictionary=concepts, clip=clip_backbone,
                      device=device, l1_penalty=0.25, return_weights=True)
splice_model.eval()

vocab_path = "/root/.cache/splice/vocab/laion.txt"
with open(vocab_path) as f:
    vocab = [l.strip() for l in f.readlines()]
vocab = vocab[-10000:]
print("SpLiCE OK")


In [ ]:
VOC_SYNONYMS = {
    "aeroplane":   ["airplane", "aircraft", "plane", "planes", "jet", "jets",
                    "aviation", "airline", "airlines", "airplanes"],
    "bicycle":     ["bicycle", "bicycles", "bike", "bikes", "cycling", "cyclist", "cyclists"],
    "bird":        ["bird", "birds", "sparrow", "eagle", "robin", "parrot", "owl"],
    "boat":        ["boat", "boats", "ship", "ships", "vessel", "yacht", "sailing", "canoe", "kayak", "ferry"],
    "bottle":      ["bottle", "bottles", "flask", "jar", "jug"],
    "bus":         ["bus", "buses", "coach", "coaches", "shuttle", "transit"],
    "car":         ["car", "cars", "automobile", "vehicle", "vehicles", "sedan", "auto", "autos", "coupe", "hatchback", "suv"],
    "cat":         ["cat", "cats", "kitten", "kittens", "kitty"],
    "chair":       ["chair", "chairs", "seat", "seats", "stool", "seating"],
    "cow":         ["cow", "cows", "cattle", "bull", "bulls", "ox", "calf", "dairy"],
    "diningtable": ["dining", "table", "tables", "desk"],
    "dog":         ["dog", "dogs", "puppy", "puppies", "canine", "hound", "pup"],
    "horse":       ["horse", "horses", "pony", "stallion", "mare"],
    "motorbike":   ["motorbike", "motorcycle", "motorcycles"],
    "person":      ["person", "persons", "people", "man", "woman", "men", "women",
                    "human", "humans", "boy", "girl", "child", "children", "individual"],
    "pottedplant": ["plant", "plants", "potted", "flower", "flowers", "succulent"],
    "sheep":       ["sheep", "lamb", "ram", "flock", "wool"],
    "sofa":        ["sofa", "couch", "sectional"],
    "train":       ["train", "trains", "locomotive", "rail", "railway", "railroad", "metro", "subway"],
    "tvmonitor":   ["tv", "television", "monitor", "screen", "display"],
}

## Hilfsfunktionen
- `get_gt_labels`: Liest die Ground-Truth-Klassen eines Bildes aus der VOC XML-Annotation
- `get_bucket`: Ordnet ein Objekt anhand seiner relativen Fläche einem Größen-Bucket zu
- `get_top_concepts`: Encodiert ein Bild-Tensor mit SpLiCE und gibt die Top-k Konzepte zurück
- `is_found`: Prüft ob ein GT-Label in den Top-Konzepten vorkommt (Found-Metrik)


In [ ]:
# Hilfsfunktionen
def get_gt_boxes_and_labels(ann_path):
    tree = ET.parse(ann_path)
    root = tree.getroot()
    objects = []
    for obj in root.findall("object"):
        label = obj.find("name").text
        bb = obj.find("bndbox")
        box = torch.tensor([
            int(bb.find("xmin").text), int(bb.find("ymin").text),
            int(bb.find("xmax").text), int(bb.find("ymax").text),
        ])
        objects.append((box, label))
    return objects

def get_bucket_voc(relative_size):
    if relative_size < 0.03:
        return "1 | sehr klein (1-5%)"
    elif relative_size < 0.05:
        return "2 | klein (5-10%)"
    elif relative_size < 0.10:
        return "3 | mittel (10-20%)"
    elif relative_size < 0.20:
        return "4 | groß (20-30%)"
    else:
        return "5 | sehr groß (30-85%)"

def get_bucket_coco(relative_size):
    if relative_size < 0.03:
        return "1 | sehr klein (1-3%)"
    elif relative_size < 0.05:
        return "2 | klein (3-5%)"
    elif relative_size < 0.10:
        return "3 | mittel (5-10%)"
    elif relative_size < 0.20:
        return "4 | groß (10-20%)"
    else:
        return "5 | sehr groß (20-85%)"

def get_top_concepts(image_tensor, k=100):
    processed = splice_preprocess(image_tensor.unsqueeze(0)).to(device)
    with torch.no_grad():
        weights, _ = splice_model.encode_image(processed)
    top_idx = weights[0].topk(k).indices.tolist()
    return [vocab[i] for i in top_idx]

def expand_label(gt_label, synonym_dict):
    return synonym_dict.get(gt_label, [gt_label])

# Überprüft, ob das Ground-Truth-Label (oder ein Synonym) in den Top-K-Konzepten gefunden wird
def is_found(gt_label, top_concepts, synonym_dict):
    expanded = expand_label(gt_label, synonym_dict)
    return any(lbl in w or w in lbl for w in top_concepts for lbl in expanded)


## Bucket-Analyse Loop
Für jedes Bild im VOC-Subset (300 Bilder):
1. GT-Labels aus Annotation laden
2. MaskRCNN erkennt Objekte → Bounding Boxes
3. Jede Box wird nach relativer Größe einem Bucket zugeordnet
4. Box wird mit **Crop** und **Circle** encodiert
5. Found-Metrik wird pro Bucket akkumuliert


In [ ]:
from utils import draw_circle_on_image

VOC_IMG_DIR = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/JPEGImages"
VOC_ANN_DIR = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/Annotations"
IMGSET      = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/ImageSets/Main/val.txt"

with open(IMGSET) as f:
    image_ids = [l.strip() for l in f if l.strip()][:1000]

results = defaultdict(lambda: {"crop_found": 0, "circle_found": 0, "total": 0})

for img_id in image_ids:
    img_path = os.path.join(VOC_IMG_DIR, img_id + ".jpg")
    ann_path = os.path.join(VOC_ANN_DIR, img_id + ".xml")
    if not os.path.exists(img_path) or not os.path.exists(ann_path):
        continue

    image = TF.to_tensor(Image.open(img_path).convert("RGB")).to(device)
    H, W = image.shape[1], image.shape[2]
    image_area = H * W

    for box, label in get_gt_boxes_and_labels(ann_path):
        x1, y1, x2, y2 = box.long()
        box_area = ((x2 - x1) * (y2 - y1)).item()
        rel_size = box_area / image_area

        if rel_size < 0.01 or rel_size > 0.85:
            continue

        bucket = get_bucket_voc(rel_size)

        # Crop
        crop = image[:, y1:y2, x1:x2]
        if crop.numel() == 0:
            continue
        crop_concepts = get_top_concepts(crop)

        # Circle
        circle_pil = draw_circle_on_image(image.cpu(), box.cpu())
        circle_tensor = TF.to_tensor(circle_pil)
        circle_concepts = get_top_concepts(circle_tensor)

        results[bucket]["crop_found"]   += int(is_found(label, crop_concepts, VOC_SYNONYMS))
        results[bucket]["circle_found"] += int(is_found(label, circle_concepts, VOC_SYNONYMS))
        results[bucket]["total"] += 1

print("Fertig")


## Ergebnisse
Found-Rate pro Größen-Bucket für Crop vs. Circle.
Ziel: Herausfinden ab welcher Objektgröße Circle besser als Crop abschneidet → Schwellenwert für adaptives Marking.

Total = Anzahl der GT-Objekt-Instanzen (einzelne Boxen) in diesem Bucket über alle 1000 Bilder. 

In [ ]:
print(f"{'Bucket':<25} {'Crop Found':>12} {'Circle Found':>14}  {'Obj. Instanzen':>16}")
print("-" * 90)
for bucket in ["1 | sehr klein (1-5%)", "2 | klein (5-10%)", 
               "3 | mittel (10-20%)", "4 | groß (20-30%)", 
               "5 | sehr groß (30-85%)"]:
    r = results[bucket]
    n = r["total"]
    if n == 0:
        continue
    crop_rate = r["crop_found"] / n
    circle_rate = r["circle_found"] / n
    crop_abs = r["crop_found"]
    circle_abs = r["circle_found"]
    print(f"{bucket:<25} {crop_rate:>11.3f} {circle_rate:>13.3f} {n:>16}")

## COCO-Subset (N=500)
Gleiche Bucket-Analyse auf COCO val2017 — GT-Boxen aus JSON-Annotationen.


In [ ]:
COCO_SYNONYMS = {
    "person":        ["person", "persons", "people", "man", "woman", "men", "women",
                      "human", "humans", "boy", "girl", "child", "children", "individual"],
    "bicycle":       ["bicycle", "bicycles", "bike", "bikes", "cycling", "cyclist", "cyclists"],
    "car":           ["car", "cars", "automobile", "vehicle", "vehicles", "sedan",
                      "auto", "autos", "coupe", "hatchback", "suv"],
    "motorcycle":    ["motorcycle", "motorcycles", "motorbike"],
    "airplane":      ["airplane", "airplanes", "aircraft", "plane", "planes", "jet", "jets",
                      "aviation", "airline", "airlines"],
    "bus":           ["bus", "buses", "coach", "coaches", "shuttle", "transit"],
    "train":         ["train", "trains", "locomotive", "rail", "railway", "railroad", "metro", "subway"],
    "truck":         ["truck", "trucks", "lorry", "van", "pickup", "freight"],
    "boat":          ["boat", "boats", "ship", "ships", "vessel", "yacht", "sailing",
                      "canoe", "kayak", "ferry"],
    "traffic light": ["traffic", "signal", "intersection"],
    "fire hydrant":  [],
    "stop sign":     ["sign", "signs", "signage"],
    "parking meter": ["parking", "meter"],
    "bench":         ["bench", "benches", "seat", "seats", "seating"],
    "bird":          ["bird", "birds", "sparrow", "eagle", "robin", "parrot", "owl"],
    "cat":           ["cat", "cats", "kitten", "kittens", "kitty"],
    "dog":           ["dog", "dogs", "puppy", "puppies", "canine", "hound", "pup"],
    "horse":         ["horse", "horses", "pony", "stallion", "mare"],
    "sheep":         ["sheep", "lamb", "ram", "flock", "wool"],
    "cow":           ["cow", "cows", "cattle", "bull", "bulls", "ox", "calf", "dairy"],
    "elephant":      ["elephant", "elephants"],
    "bear":          ["bear", "bears", "grizzly", "polar", "panda"],
    "zebra":         ["zebra"],
    "giraffe":       ["giraffe"],
    "backpack":      ["backpack", "backpacks", "bag", "bags"],
    "umbrella":      ["umbrella", "umbrellas"],
    "handbag":       ["handbag", "handbags", "purse", "bag", "bags"],
    "tie":           ["tie", "ties"],
    "suitcase":      ["suitcase", "luggage", "baggage"],
    "frisbee":       ["disc", "disk"],
    "skis":          ["skis", "ski", "skiing", "skier"],
    "snowboard":     ["snowboard"],
    "sports ball":   ["ball", "balls", "soccer", "football", "basketball", "baseball"],
    "kite":          ["kite"],
    "baseball bat":  ["bat", "bats", "baseball"],
    "baseball glove":["glove", "gloves", "baseball"],
    "skateboard":    ["skateboard", "skater"],
    "surfboard":     ["surfing", "surfer"],
    "tennis racket": ["tennis", "racket"],
    "bottle":        ["bottle", "bottles", "flask", "jar", "jug"],
    "wine glass":    ["wine", "glass", "glasses"],
    "cup":           ["cup", "cups", "mug", "mugs"],
    "fork":          ["fork", "cutlery"],
    "knife":         ["knife", "knives", "blade", "cutlery"],
    "spoon":         ["spoon", "spoons", "cutlery"],
    "bowl":          ["bowl", "bowls", "dish", "dishes"],
    "banana":        ["banana", "bananas"],
    "apple":         ["apple", "apples"],
    "sandwich":      ["sandwich", "sandwiches"],
    "orange":        ["orange", "oranges"],
    "broccoli":      ["broccoli"],
    "carrot":        ["carrot", "carrots"],
    "hot dog":       ["sausage"],
    "pizza":         ["pizza"],
    "donut":         ["donut", "donuts"],
    "cake":          ["cake", "cakes", "pastry"],
    "chair":         ["chair", "chairs", "seat", "seats", "stool", "seating"],
    "couch":         ["couch", "sofa", "sectional"],
    "potted plant":  ["plant", "plants", "potted", "flower", "flowers", "succulent"],
    "bed":           ["bed", "beds", "bedding", "bedroom", "mattress"],
    "dining table":  ["dining", "table", "tables", "desk"],
    "toilet":        ["toilet", "bathroom", "wc"],
    "tv":            ["tv", "television", "monitor", "screen", "display"],
    "laptop":        ["laptop", "laptops", "notebook"],
    "mouse":         ["mouse", "mice"],
    "remote":        ["remote", "controller"],
    "keyboard":      ["keyboard", "typing"],
    "cell phone":    ["phone", "phones", "smartphone", "mobile", "iphone"],
    "microwave":     ["microwave"],
    "oven":          ["oven", "stove"],
    "toaster":       [],
    "sink":          ["sink", "sinks", "basin"],
    "refrigerator":  ["refrigerator", "fridge", "freezer"],
    "book":          ["book", "books", "bookshelf", "reading"],
    "clock":         ["clock", "clocks", "watch", "timer"],
    "vase":          ["vase", "vases", "pot", "pottery"],
    "scissors":      ["scissors"],
    "teddy bear":    ["teddy", "stuffed", "plush", "toy"],
    "hair drier":    ["dryer"],
    "toothbrush":    ["toothbrush"],
}

In [ ]:
import json

IMG_DIR_COCO  = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/images/val2017"
ANN_PATH_COCO = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/annotations/instances_val2017.json"

with open(ANN_PATH_COCO) as f:
    coco_data = json.load(f)

cat_id_to_name  = {c["id"]: c["name"] for c in coco_data["categories"]}
img_id_to_fname = {img["id"]: img["file_name"] for img in coco_data["images"]}

img_id_to_anns = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    x, y, w, h = ann["bbox"]
    img_id_to_anns.setdefault(iid, []).append((
        torch.tensor([x, y, x + w, y + h]),
        cat_id_to_name[ann["category_id"]]
    ))

subset_img_ids = [
    img["id"] for img in coco_data["images"]
    if img["id"] in img_id_to_anns
][:500]
print(f"Subset: {len(subset_img_ids)} Bilder")


In [ ]:
results_coco = defaultdict(lambda: {"crop_found": 0, "circle_found": 0, "total": 0})

for img_id in subset_img_ids:
    img_path = os.path.join(IMG_DIR_COCO, img_id_to_fname[img_id])
    if not os.path.exists(img_path):
        continue

    image = TF.to_tensor(Image.open(img_path).convert("RGB")).to(device)
    H, W = image.shape[1], image.shape[2]
    image_area = H * W

    for box, label in img_id_to_anns[img_id]:
        x1, y1, x2, y2 = box.long()
        box_area = ((x2 - x1) * (y2 - y1)).item()
        rel_size = box_area / image_area

        if rel_size < 0.01 or rel_size > 0.85:
            continue

        bucket = get_bucket_coco(rel_size)

        crop = image[:, y1:y2, x1:x2]
        if crop.numel() == 0:
            continue
        crop_concepts = get_top_concepts(crop)

        circle_pil = draw_circle_on_image(image.cpu(), box.cpu())
        circle_tensor = TF.to_tensor(circle_pil)
        circle_concepts = get_top_concepts(circle_tensor)

        results_coco[bucket]["crop_found"]   += int(is_found(label, crop_concepts, COCO_SYNONYMS))
        results_coco[bucket]["circle_found"] += int(is_found(label, circle_concepts, COCO_SYNONYMS))
        results_coco[bucket]["total"] += 1

print("Fertig")


In [ ]:
print(f"{'Bucket':<25} {'Crop Found':>12} {'Circle Found':>14} {'Obj. Instanzen':>16}")
print("-" * 90)
for bucket in ["1 | sehr klein (1-3%)", "2 | klein (3-5%)", "3 | mittel (5-10%)", 
               "4 | groß (10-20%)", "5 | sehr groß (20-85%)"]:
    r = results_coco[bucket]
    n = r["total"]
    if n == 0:
        continue
    crop_rate = r["crop_found"] / n
    circle_rate = r["circle_found"] / n

    print(f"{bucket:<25} {crop_rate:>11.3f} {circle_rate:>13.3f} {n:>16}")
